# Paredes por píxel

> ¿El modelo encuentra las paredes, pero las corta distinto que la etiqueta? En cada plano junta todas las paredes en un mapa de píxeles, uno predicho y otro etiquetado, y los compara en FloorPlanCAD y en BLD-AR.

In [1]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

Tue Nov 10 12:51:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 Ti     Off |   00000000:01:00.0 Off |                  N/A |
| 30%   44C    P2             45W /  285W |    4446MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8

## Imports

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image_as_pil

## Rutas y parámetros

In [3]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
FLOORPLANCAD = REPO / "datasets/floor-plan-cad-yolo"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.25  # desde esta confianza una predicción cuenta como pared
DEVICE = "cuda:0"

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Modelo

In [4]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Mapas de pared y medidas

In [5]:
def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def pixel_map(size, polygons):
    """Mapa del tamaño del plano, True dentro de los polígonos (listas x, y, x, y, ...)."""
    canvas = Image.new("1", size)
    draw = ImageDraw.Draw(canvas)
    for polygon in polygons:
        draw.polygon(polygon, fill=1)
    return np.array(canvas)


def predicted_walls(image_path):
    """Polígonos de wall que predice el modelo por tiles, como en la evaluación en BLD-AR."""
    result = get_sliced_prediction(
        str(image_path), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
    )
    return [polygon for prediction in result.to_coco_predictions()
            if prediction["category_name"] == "wall" for polygon in prediction["segmentation"]]


def overlap(predicted, labeled):
    """Coincidencia, pared encontrada y pared bien marcada de un plano."""
    both = (predicted & labeled).sum()
    well_marked = both / predicted.sum() if predicted.any() else np.nan
    return both / (predicted | labeled).sum(), both / labeled.sum(), well_marked

## FloorPlanCAD (test)

In [6]:
names = yaml.safe_load((FLOORPLANCAD / "floor-plan-cad-yolo.yaml").read_text())["names"]
wall_id = next(i for i, name in names.items() if name == "wall")

floorplancad = []
for image_path in sorted((FLOORPLANCAD / "images/test").iterdir()):
    size = Image.open(image_path).size
    rows = [line.split() for line in (FLOORPLANCAD / "labels/test" / f"{image_path.stem}.txt").read_text().splitlines()]
    # Etiqueta YOLO: clase y puntos x, y entre 0 y 1
    walls = [(np.reshape(np.array(row[1:], dtype=float), (-1, 2)) * size).ravel().tolist()
             for row in rows if int(row[0]) == wall_id]
    if walls:
        floorplancad.append(overlap(pixel_map(size, predicted_walls(image_path)), pixel_map(size, walls)))

## BLD-AR

> Solo se mide wall: lo que el modelo predice como pared sobre las demás clases de BLD-AR (puertas, ventanas, columnas, vacíos…) no cuenta.

In [7]:
def bldar_maps(json_path):
    """Mapas de pared etiquetada y predicha de un plano de BLD-AR; None si no tiene paredes."""
    data = json.loads(json_path.read_text())
    size = (data["imageWidth"], data["imageHeight"])
    walls = [shape_points(shape).ravel().tolist() for shape in data["shapes"] if shape["label"] == "wall"]
    others = [shape_points(shape).ravel().tolist() for shape in data["shapes"] if shape["label"] != "wall"]
    if not walls:
        return None
    labeled = pixel_map(size, walls)
    predicted = pixel_map(size, predicted_walls(BLDAR / data["imagePath"]))
    predicted &= labeled | ~pixel_map(size, others)  # se conserva donde la etiqueta es pared
    return labeled, predicted


bldar = {}
for json_path in sorted(BLDAR.glob("*.json")):
    maps = bldar_maps(json_path)
    if maps:
        labeled, predicted = maps
        bldar[json_path.stem] = overlap(predicted, labeled)

## Resultado

> Promedio por plano, en los planos con paredes etiquetadas. La hipótesis se confirma si la pared encontrada en BLD-AR llega al 80 % o más de la de FloorPlanCAD, y se descarta si queda debajo del 50 %. No se usa la coincidencia porque FloorPlanCAD tiene muros sin etiquetar que el modelo marca.

In [8]:
results = {"FloorPlanCAD": floorplancad, "BLD-AR": list(bldar.values())}
means = {name: np.nanmean(values, axis=0) for name, values in results.items()}
ratio = means["BLD-AR"] / means["FloorPlanCAD"]

table = "| | Planos | Coincidencia | Pared encontrada | Pared bien marcada |\n|---|---|---|---|---|\n"
for name, mean in means.items():
    table += f"| {name} | {len(results[name])} | {mean[0]:.3f} | {mean[1]:.3f} | {mean[2]:.3f} |\n"
table += f"| BLD-AR / FloorPlanCAD | | {ratio[0]:.0%} | {ratio[1]:.0%} | {ratio[2]:.0%} |"
display(Markdown(f"**{WEIGHTS.parent.name} · confianza {CONF}**\n\n" + table))

**00-YOLO26m-SEG-1024 · confianza 0.25**

| | Planos | Coincidencia | Pared encontrada | Pared bien marcada |
|---|---|---|---|---|
| FloorPlanCAD | 1333 | 0.713 | 0.812 | 0.860 |
| BLD-AR | 56 | 0.572 | 0.611 | 0.876 |
| BLD-AR / FloorPlanCAD | | 80% | 75% | 102% |

## Revisión plano por plano

> Recorre los planos de BLD-AR de mayor a menor pared encontrada (o al revés, con la línea comentada). Cada vez que se ejecuta la celda de "Siguiente plano", muestra uno: a la izquierda la pared etiquetada y a la derecha la predicha que cuenta en la medida.

In [ ]:
def reduced(mask, size):
    """Mapa reducido a size; un píxel queda marcado si tenía algo de pared."""
    return np.array(Image.fromarray(mask).convert("L").resize(size, Image.Resampling.BOX)) > 0


def painted(picture, mask, color):
    """Plano con la pared pintada encima, semitransparente."""
    rgb = np.array(picture.convert("RGB"), dtype=float)
    rgb[mask] = 0.5 * rgb[mask] + 0.5 * np.array(color)
    return rgb.astype(np.uint8)


# Del mejor al peor; para ir del peor al mejor, reemplazar por la línea comentada
order = iter(enumerate(sorted(bldar, key=lambda name: bldar[name][1], reverse=True), start=1))
# order = iter(enumerate(sorted(bldar, key=lambda name: bldar[name][1]), start=1))

## Siguiente plano

In [ ]:
position, name = next(order)
labeled, predicted = bldar_maps(BLDAR / f"{name}.json")
data = json.loads((BLDAR / f"{name}.json").read_text())
picture = read_image_as_pil(str(BLDAR / data["imagePath"])).convert("L")
picture.thumbnail((3000, 3000))  # hay planos de hasta 15000 px de lado
coincidence, found, well_marked = bldar[name]

fig, axes = plt.subplots(1, 2, figsize=(20, 1 + 10 * picture.height / picture.width))
for ax, title, mask, color in zip(axes, ("Etiqueta", "Predicción"), (labeled, predicted), ((0, 170, 0), (220, 0, 0))):
    ax.imshow(painted(picture, reduced(mask, picture.size), color))
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"{position} de {len(bldar)} · {name} · coincidencia {coincidence:.2f} · "
             f"pared encontrada {found:.2f} · pared bien marcada {well_marked:.2f}")
plt.show()